# LLDP Flow-Based IDS - ML Pipeline

Random Forest classifier for LLDP attack detection in SDN environments.

References:
- github.com/arsheen/IDS-on-SDN-using-Machine-Learning
- github.com/rana-uzair-ahmed/MininetIDS
- scikit-learn.org/stable/modules/ensemble.html#random-forests

In [ ]:
BASE_PATH = '/content'

In [ ]:
import os

csv_file = f'{BASE_PATH}/FINAL_LLDP_DATASET_COMPLETE_enriched.csv'

if not os.path.exists(csv_file) or os.path.getsize(csv_file) < 10_000_000:
    print("Downloading dataset from GitHub...")
    !wget -q -O {csv_file} https://github.com/rae81/655Rami/raw/refs/heads/claude/setup-ids-project-01P3KnjqzpbcbmiAHHWsE4j6/ramifinal655/mlmodel/data/FINAL_LLDP_DATASET_COMPLETE_enriched.csv
    print("Download complete")

size_mb = os.path.getsize(csv_file) / (1024 * 1024)
print(f"Dataset file: {size_mb:.2f} MB")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, precision_recall_fscore_support
)
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')

## 1. Load and Validate Dataset

In [ ]:
csv_path = f'{BASE_PATH}/FINAL_LLDP_DATASET_COMPLETE_enriched.csv'

with open(csv_path, 'rb') as f:
    line_count = f.read().count(b'\n')

if line_count < 100000:
    raise ValueError(f"File corrupted: {line_count:,} lines (expected 108,298). Re-run download cell.")

df_raw = pd.read_csv(csv_path, low_memory=False)

print(f"Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("Validation: PASS\n")

class_dist = df_raw['label'].value_counts()
print("Class Distribution:")
for label, count in class_dist.items():
    pct = count / len(df_raw) * 100
    print(f"  {label:15s}: {count:6,} ({pct:5.1f}%)")

print(f"\nImbalance Ratio: {class_dist.max() / class_dist.min():.1f}:1")

## 2. Data Cleaning

In [ ]:
df = df_raw.drop_duplicates()
removed = df_raw.shape[0] - len(df)
print(f"Duplicates removed: {removed:,} ({removed/df_raw.shape[0]*100:.1f}%)")

df = df.drop(columns=['size_z_src', 'packet_rate_inst', 'burstiness_cv'])
print(f"Dropped sparse features: size_z_src, packet_rate_inst, burstiness_cv")

if df['inter_frame_delta'].isnull().sum() > 0:
    median_val = df['inter_frame_delta'].median()
    df['inter_frame_delta'].fillna(median_val, inplace=True)
    print(f"Imputed inter_frame_delta with median: {median_val:.6f}")

print(f"\nFinal dataset: {len(df):,} rows x {len(df.columns)} columns")

## 3. Feature Selection

In [ ]:
EXCLUDE_COLS = ['timestamp', 'time_epoch', 'eth_src', 'eth_dst', 'label']
numeric_features = [col for col in df.columns if col not in EXCLUDE_COLS]

features_to_keep = []
for feature in numeric_features:
    if df[feature].nunique() > 1:
        features_to_keep.append(feature)
    else:
        print(f"Dropped zero-variance: {feature}")

print(f"\nSelected features ({len(features_to_keep)}):")
for i, feat in enumerate(features_to_keep, 1):
    print(f"  {i:2d}. {feat}")

## 4. EDA Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

class_counts = df['label'].value_counts()
axes[0, 0].bar(range(len(class_counts)), class_counts.values, color='steelblue')
axes[0, 0].set_xticks(range(len(class_counts)))
axes[0, 0].set_xticklabels(class_counts.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Class Distribution')
axes[0, 0].grid(axis='y', alpha=0.3)

for label in df['label'].unique():
    subset = df[df['label'] == label]['ttl']
    axes[0, 1].hist(subset, bins=30, alpha=0.5, label=label)
axes[0, 1].set_xlabel('TTL Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('TTL Distribution by Class')
axes[0, 1].legend()

for label in df['label'].unique():
    subset = df[df['label'] == label]['packet_rate_win']
    axes[1, 0].hist(subset, bins=30, alpha=0.5, label=label)
axes[1, 0].set_xlabel('Packet Rate Window')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Packet Rate Distribution by Class')
axes[1, 0].legend()

tlv_counts = df.groupby('label')['tlv_count'].mean().sort_values()
axes[1, 1].barh(range(len(tlv_counts)), tlv_counts.values, color='coral')
axes[1, 1].set_yticks(range(len(tlv_counts)))
axes[1, 1].set_yticklabels(tlv_counts.index)
axes[1, 1].set_xlabel('Average TLV Count')
axes[1, 1].set_title('TLV Count by Class')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/eda_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
key_features = ['packet_size', 'ttl', 'tlv_count', 'packet_rate_win', 'ttl_dev', 'count_win']
available_key = [f for f in key_features if f in features_to_keep]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(available_key):
    df.boxplot(column=feature, by='label', ax=axes[idx])
    axes[idx].set_title(f'{feature} by Class')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel(feature)
    plt.sca(axes[idx])
    plt.xticks(rotation=45, ha='right')

for idx in range(len(available_key), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Feature Distributions by Attack Class', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Data Preparation

In [ ]:
X = df[features_to_keep].copy()
y = df['label'].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=features_to_keep, index=df.index)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")

print(f"\nTest class distribution:")
for cls, cnt in y_test.value_counts().sort_index().items():
    print(f"  {cls:15s}: {cnt:5,}")

## 6. Model Training

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Training model with GridSearchCV...\n")
start = time.time()
grid_search.fit(X_train, y_train)
train_time = time.time() - start

print(f"\nTraining time: {train_time:.2f}s")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

rf_model = grid_search.best_estimator_

## 7. Model Evaluation

In [ ]:
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
gap = train_acc - test_acc

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Overfitting Gap: {gap:.4f} ({gap*100:.2f}%)")

if gap > 0.10:
    print("Status: WARNING - High overfitting")
elif gap > 0.05:
    print("Status: Moderate overfitting")
else:
    print("Status: Good generalization")

In [ ]:
f1_macro = f1_score(y_test, y_test_pred, average='macro')
f1_weighted = f1_score(y_test, y_test_pred, average='weighted')

print(f"F1-Score (Macro):    {f1_macro:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")

class_names = sorted(y.unique())
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_test_pred, labels=class_names, zero_division=0
)

print(f"\n{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
print("-" * 60)
for i, cls in enumerate(class_names):
    print(f"{cls:<15} {precision[i]:<10.4f} {recall[i]:<10.4f} {f1[i]:<10.4f} {support[i]:<10}")

In [ ]:
fpr_per_class = {}
for cls in class_names:
    y_true_binary = (y_test == cls).astype(int)
    y_pred_binary = (y_test_pred == cls).astype(int)
    
    tn = ((y_true_binary == 0) & (y_pred_binary == 0)).sum()
    fp = ((y_true_binary == 0) & (y_pred_binary == 1)).sum()
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fpr_per_class[cls] = fpr

print("False Positive Rate (FPR):")
for cls, fpr in fpr_per_class.items():
    status = "GOOD" if fpr < 0.05 else "OK" if fpr < 0.10 else "HIGH"
    print(f"{cls:<15}: {fpr:.4f} ({fpr*100:.2f}%) [{status}]")

avg_fpr = np.mean(list(fpr_per_class.values()))
print(f"\nAverage FPR: {avg_fpr:.4f} ({avg_fpr*100:.2f}%)")

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=class_names)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - LLDP Attack Classification', fontsize=14)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Detection Latency

In [ ]:
n_samples = min(1000, len(X_test))
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)
X_latency = X_test.iloc[sample_indices]

latencies = []
for i in range(len(X_latency)):
    sample = X_latency.iloc[i:i+1]
    start = time.perf_counter()
    _ = rf_model.predict(sample)
    latencies.append((time.perf_counter() - start) * 1000)

latencies = np.array(latencies)

print("Detection Latency:")
print(f"  Mean:   {latencies.mean():.4f} ms")
print(f"  Median: {np.median(latencies):.4f} ms")
print(f"  95th:   {np.percentile(latencies, 95):.4f} ms")
print(f"  99th:   {np.percentile(latencies, 99):.4f} ms")

batch_size = min(1000, len(X_test))
X_batch = X_test.iloc[:batch_size]
start = time.perf_counter()
_ = rf_model.predict(X_batch)
throughput = batch_size / (time.perf_counter() - start)

print(f"\nThroughput: {throughput:,.0f} events/second")

## 10. Feature Importance

In [ ]:
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'feature': features_to_keep,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Save Model

In [ ]:
joblib.dump(rf_model, f'{BASE_PATH}/lldp_rf_model.pkl')
joblib.dump(scaler, f'{BASE_PATH}/feature_scaler.pkl')
pd.DataFrame({'feature': features_to_keep}).to_csv(f'{BASE_PATH}/feature_list.csv', index=False)

print("Model artifacts saved:")
print(f"  - lldp_rf_model.pkl")
print(f"  - feature_scaler.pkl")
print(f"  - feature_list.csv")

loaded_model = joblib.load(f'{BASE_PATH}/lldp_rf_model.pkl')
test_pred = loaded_model.predict(X_test.iloc[:1])
print(f"\nModel loading verified: {test_pred[0]}")

## 12. Summary

In [ ]:
print("=" * 70)
print("LLDP IDS MODEL SUMMARY")
print("=" * 70)
print(f"\nDataset: {len(df):,} samples, {len(class_names)} classes")
print(f"Features: {len(features_to_keep)}")
print(f"Training time: {train_time:.2f}s")
print(f"\nPerformance:")
print(f"  Accuracy:     {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  F1-Macro:     {f1_macro:.4f}")
print(f"  Avg FPR:      {avg_fpr:.4f} ({avg_fpr*100:.2f}%)")
print(f"  Latency:      {latencies.mean():.4f} ms")
print(f"  Throughput:   {throughput:,.0f} events/sec")
print(f"\nOverfitting:  {gap:.4f} ({'GOOD' if gap < 0.05 else 'WARNING'})")
print("\n" + "=" * 70)